In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- es_update_pivot_join ---
FIX_ES_UPDATE_PIVOT_JOIN_ALL_RESPONSES_PD = pd.DataFrame({"realization": [1, 2, 3], "response": [10.0, 20.0, 30.0]})
FIX_ES_UPDATE_PIVOT_JOIN_OBSERVATION_PD = pd.DataFrame({"realization": [1, 2, 3], "observations": [5.0, 6.0, 7.0], "std": [0.1, 0.1, 0.1]})
FIX_ES_UPDATE_PIVOT_JOIN_ALL_RESPONSES_PL = pl.from_pandas(FIX_ES_UPDATE_PIVOT_JOIN_ALL_RESPONSES_PD)
FIX_ES_UPDATE_PIVOT_JOIN_OBSERVATION_PL = pl.from_pandas(FIX_ES_UPDATE_PIVOT_JOIN_OBSERVATION_PD)
FIX_ES_UPDATE_PIVOT_JOIN_ALL_RESPONSES = FIX_ES_UPDATE_PIVOT_JOIN_ALL_RESPONSES_PD
FIX_ES_UPDATE_PIVOT_JOIN_OBSERVATION = FIX_ES_UPDATE_PIVOT_JOIN_OBSERVATION_PD

# --- es_update_scaling_factors ---
class ScalingFactorEnsemble:
    def __init__(self):
        self.saved = None
    def save_observation_scaling_factors(self, value):
        self.saved = value

FIX_ES_UPDATE_SCALING_FACTORS_ENSEMBLE = ScalingFactorEnsemble()
FIX_ES_UPDATE_SCALING_FACTORS_SCALING_FACTORS_DFS_PD = [
    pd.DataFrame({"input_group": ["g1"], "obs_key": ["k1"], "index": [0], "factor": [1.0]}),
    pd.DataFrame({"input_group": ["g2"], "obs_key": ["k2"], "index": [1], "factor": [0.9]}),
]
FIX_ES_UPDATE_SCALING_FACTORS_SCALING_FACTORS_DFS_PL = [pl.from_pandas(df) for df in FIX_ES_UPDATE_SCALING_FACTORS_SCALING_FACTORS_DFS_PD]
FIX_ES_UPDATE_SCALING_FACTORS_SCALING_FACTORS_DFS = FIX_ES_UPDATE_SCALING_FACTORS_SCALING_FACTORS_DFS_PD

print("✅ Fixtures loaded")


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_es_update_pivot_join(all_responses, observation):
    oar = observation.merge(all_responses, on=["realization"], how="left")
    return {
    "obs_keys_count": len(oar["observations"]),
    "obs_values":     oar["observations"].values.ravel(),
    "obs_errors":     oar["std"].values.ravel(),
    "n_reals":        len(oar["realization"]),
    }
    return oar

def before_es_update_scaling_factors(ensemble, scaling_factors_dfs):
    scaling_factors_df = pd.concat(scaling_factors_dfs).set_index(
        ["input_group", "obs_key", "index"], verify_integrity=True
    )
    ensemble.save_observation_scaling_factors(scaling_factors_df.to_xarray())
    return scaling_factors_df

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_es_update_pivot_join(all_responses, observation):

    oar = observation.join(all_responses, on="realization", how="left")
    return {
        "obs_keys_count": len(oar["observations"]),
        "obs_values":     oar["observations"].to_numpy().ravel(),
        "obs_errors":     oar["std"].to_numpy().ravel(),
        "n_reals":        len(oar["realization"]),
    }

def gen_es_update_scaling_factors(ensemble, scaling_factors_dfs):

    scaling_factors_df = pl.concat(scaling_factors_dfs, how="vertical").to_pandas()
    if scaling_factors_df.duplicated(subset=["input_group", "obs_key", "index"]).any():
        raise ValueError("Index has duplicate keys")

    scaling_factors_df = scaling_factors_df.set_index(
        ["input_group", "obs_key", "index"], verify_integrity=True
    )
    ensemble.save_observation_scaling_factors(scaling_factors_df.to_xarray())
    return scaling_factors_df

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: es_update_scaling_factors ===

# L1 smoke – generated
try:
    _r = gen_es_update_scaling_factors(ScalingFactorEnsemble(), FIX_ES_UPDATE_SCALING_FACTORS_SCALING_FACTORS_DFS_PL)
    print("✅ L1 smoke gen_es_update_scaling_factors: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_es_update_scaling_factors: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_es_update_scaling_factors(ScalingFactorEnsemble(), FIX_ES_UPDATE_SCALING_FACTORS_SCALING_FACTORS_DFS_PD)
    print("✅ L1 smoke before_es_update_scaling_factors: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_es_update_scaling_factors: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_es_update_scaling_factors(ScalingFactorEnsemble(), FIX_ES_UPDATE_SCALING_FACTORS_SCALING_FACTORS_DFS_PD)
    _rg = gen_es_update_scaling_factors(ScalingFactorEnsemble(), FIX_ES_UPDATE_SCALING_FACTORS_SCALING_FACTORS_DFS_PL)
    compare(_rb.reset_index(), _rg, "es_update_scaling_factors")
except Exception as _e:
    print(f"❌ L2 equivalence es_update_scaling_factors: setup error — {type(_e).__name__}: {_e}")

# L3 edge – duplicate multi-index is rejected by both implementations
try:
    _dup_pd = [pd.DataFrame({"input_group": ["g1"], "obs_key": ["k1"], "index": [0], "factor": [1.0]}),
               pd.DataFrame({"input_group": ["g1"], "obs_key": ["k1"], "index": [0], "factor": [0.5]})]
    _dup_pl = [pl.from_pandas(df) for df in _dup_pd]
    try:
        before_es_update_scaling_factors(ScalingFactorEnsemble(), _dup_pd)
        _before_error = None
    except Exception as _e:
        _before_error = type(_e).__name__
    try:
        gen_es_update_scaling_factors(ScalingFactorEnsemble(), _dup_pl)
        _gen_error = None
    except Exception as _e:
        _gen_error = type(_e).__name__
    if _before_error == _gen_error == "ValueError":
        print("✅ L3 edge es_update_scaling_factors duplicate index: MATCH")
    else:
        print(f"❌ L3 edge es_update_scaling_factors duplicate index: MISMATCH — before={_before_error}, gen={_gen_error}")
except Exception as _e:
    print(f"❌ L3 edge es_update_scaling_factors: {type(_e).__name__}: {_e}")

# AUDIT-69: compare the value sent to the ensemble as well as the return value.
try:
    _before_ensemble = ScalingFactorEnsemble(); _gen_ensemble = ScalingFactorEnsemble()
    before_es_update_scaling_factors(_before_ensemble, FIX_ES_UPDATE_SCALING_FACTORS_SCALING_FACTORS_DFS_PD)
    gen_es_update_scaling_factors(_gen_ensemble, FIX_ES_UPDATE_SCALING_FACTORS_SCALING_FACTORS_DFS_PL)
    def _saved_to_frame(value):
        if isinstance(value, pl.DataFrame): return value
        if hasattr(value, "to_dataframe"): return value.to_dataframe().reset_index()
        return value
    compare(_saved_to_frame(_before_ensemble.saved), _saved_to_frame(_gen_ensemble.saved), "es_update_scaling_factors saved value", check_row_order=True)
except Exception as _e:
    print(f"❌ L2 equivalence es_update_scaling_factors saved value: {type(_e).__name__}: {_e}")


/var/folders/bj/46g7cgnj5nj8tq01jz71cbm00000gn/T/ipykernel_31069/3993571021.py:19: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  scaling_factors_df = scaling_factors_df.set_index(
/var/folders/bj/46g7cgnj5nj8tq01jz71cbm00000gn/T/ipykernel_31069/565002479.py:14: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  scaling_factors_df = pd.concat(scaling_factors_dfs).set_index(
/var/folders/bj/46g7cgnj5nj8tq01jz71cbm00000gn/T/ipykernel_31069/565002479.py:14: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  scaling_factors_df = pd.concat(scaling_factors_dfs).set_index(
/var/folders/bj/46g7cgnj5nj8tq01jz71cbm000

✅ L1 smoke gen_es_update_scaling_factors: OK, type= DataFrame
✅ L1 smoke before_es_update_scaling_factors: OK
✅ L2 equivalence es_update_scaling_factors: MATCH
✅ L3 edge es_update_scaling_factors duplicate index: MATCH
✅ L2 equivalence es_update_scaling_factors saved value: MATCH


/var/folders/bj/46g7cgnj5nj8tq01jz71cbm00000gn/T/ipykernel_31069/565002479.py:14: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  scaling_factors_df = pd.concat(scaling_factors_dfs).set_index(
/var/folders/bj/46g7cgnj5nj8tq01jz71cbm00000gn/T/ipykernel_31069/565002479.py:14: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  scaling_factors_df = pd.concat(scaling_factors_dfs).set_index(
/var/folders/bj/46g7cgnj5nj8tq01jz71cbm00000gn/T/ipykernel_31069/3993571021.py:19: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  scaling_factors_df = scaling_factors_df.set_index(
